# SRL

In [1]:
import os
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# =========================================
# 1) PATHS
# =========================================
# site hyperspectral image (ENVI binary file or tif)
site_img = r"E:\wenqu\2024_data\site6\site6_simulation_4"

# folder containing PFT cover tif files
pft_folder = r"D:\wenqu\2024_data\PFT_Map"

In [2]:
# output trait image
out_tif = r"D:\wenqu\2024_data\root_trait_map\site6_SRL4_2021.tif"

# =========================================
# 2) SELECTED VARIABLES
# =========================================
var_list = [
    'd41', 'd44', 'd100', 'd31', 'd37', 'd25', 'd16', 'd53', 'd59', 'd92',
    'M_relative_abund_sum', 'd96', 'd112', 'd33', 'd83', 'd101', 'd24',
    'd78', 'd39', 'd35', 'd18', 'd58', 'd115', 'd30', 'd13', 'd91', 'd47',
    'd74', 'd34', 'd86', 'd117', 'G_relative_abund_sum', 'd19', 'd110',
    'd22', 'd57', 'd46', 'd75', 'd32', 'd52', 'd108', 'F_relative_abund_sum',
    'd6'
]

coefficients = np.array([
    18.33330036,  15.13791979,   8.46167696,  -2.64690847,
    14.66524473, -11.42776874, -17.47870856,   8.60347264,
    13.41634978, -10.58231367,   9.95864673,  10.23540009,
    -3.0634247 ,  14.32253839,  -3.15062394,   2.96992266,
     2.2278167 , -19.69312903,  -9.57615748, -20.94719724,
    13.93916195,  19.88160816,   1.26948231,  -7.93966447,
     4.63381007,  -5.85984001, -15.0414025 ,  -5.53938564,
   -18.53874892,  10.16458728,  -7.87416597,  15.24719633,
    12.20013923, -29.07572359, -10.81218818,  -0.49490199,
   -16.03870794,  20.79155442, -25.51458522,  12.99986561,
    17.81258042,  -6.98949352,  19.705342
], dtype=np.float32)

intercept = 119.05814089

if len(var_list) != len(coefficients):
    raise ValueError(f"var_list has {len(var_list)} variables but coefficients has {len(coefficients)} values.")

# =========================================
# 3) PFT FILE NAME MAP
#    Edit the filenames here if needed
# =========================================
pft_map = {
    "M_relative_abund_sum": os.path.join(pft_folder, "site6_M4_2021.tif"),
    "G_relative_abund_sum": os.path.join(pft_folder, "site6_G4_2021.tif"),
    "F_relative_abund_sum": os.path.join(pft_folder, "site6_F4_2021.tif"),
}

for k, v in pft_map.items():
    if not os.path.exists(v):
        raise FileNotFoundError(f"Missing PFT tif for {k}: {v}")

# =========================================
# 4) OPEN SITE IMAGE AS REFERENCE
# =========================================
with rasterio.open(site_img) as src:
    print("Site image:", site_img)
    print("Driver:", src.driver)
    print("Band count:", src.count)
    print("Shape:", src.height, src.width)

    ref_meta = src.meta.copy()
    ref_crs = src.crs
    ref_transform = src.transform
    ref_height = src.height
    ref_width = src.width
    ref_nodata = src.nodata

    # use band 1 to create initial valid mask
    ref_band = src.read(1).astype(np.float32)

    if ref_nodata is not None:
        valid_mask = (ref_band != ref_nodata) & np.isfinite(ref_band)
    else:
        valid_mask = np.isfinite(ref_band)

    # initialize output
    result = np.full((ref_height, ref_width), intercept, dtype=np.float32)

    # =========================================
    # 5) LOOP THROUGH VARIABLES
    # =========================================
    for var_name, coef in zip(var_list, coefficients):
        print(f"Processing {var_name}  coef = {coef}")

        # -----------------------------
        # derivative band from site image
        # d41 = b42 - b41
        # -----------------------------
        if var_name.startswith("d"):
            d_idx = int(var_name[1:])   # e.g. 41
            b1 = d_idx
            b2 = d_idx + 1

            if b1 < 1 or b2 > src.count:
                raise ValueError(f"{var_name} needs bands {b1} and {b2}, but site image has only {src.count} bands.")

            arr1 = src.read(b1).astype(np.float32)
            arr2 = src.read(b2).astype(np.float32)

            deriv = arr2 - arr1

            if ref_nodata is not None:
                band_mask = (arr1 != ref_nodata) & (arr2 != ref_nodata)
            else:
                band_mask = np.isfinite(arr1) & np.isfinite(arr2)

            band_mask &= np.isfinite(deriv)
            valid_mask &= band_mask

            result += deriv * coef

        # -----------------------------
        # PFT raster
        # -----------------------------
        else:
            tif_path = pft_map[var_name]

            with rasterio.open(tif_path) as pft_src:
                pft_arr = pft_src.read(1).astype(np.float32)
                pft_nodata = pft_src.nodata

                if pft_nodata is not None:
                    pft_arr[pft_arr == pft_nodata] = np.nan

                dst_arr = np.full((ref_height, ref_width), np.nan, dtype=np.float32)

                reproject(
                    source=pft_arr,
                    destination=dst_arr,
                    src_transform=pft_src.transform,
                    src_crs=pft_src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                band_mask = np.isfinite(dst_arr)
                valid_mask &= band_mask

                result += np.nan_to_num(dst_arr, nan=0.0) * coef

# =========================================
# 6) SET OUTPUT NODATA
# =========================================
out_nodata = -9999.0
result[~valid_mask] = out_nodata

# =========================================
# 7) WRITE OUTPUT
# =========================================
out_meta = ref_meta.copy()
out_meta.update({
    "driver": "GTiff",
    "count": 1,
    "dtype": "float32",
    "nodata": out_nodata
})

os.makedirs(os.path.dirname(out_tif), exist_ok=True)

with rasterio.open(out_tif, "w", **out_meta) as dst:
    dst.write(result, 1)

print("Done.")
print("Output saved to:", out_tif)

Site image: E:\wenqu\2024_data\site6\site6_simulation_4
Driver: ENVI
Band count: 122
Shape: 3152 3052
Processing d41  coef = 18.33329963684082
Processing d44  coef = 15.137919425964355
Processing d100  coef = 8.461676597595215
Processing d31  coef = -2.6469085216522217
Processing d37  coef = 14.665245056152344
Processing d25  coef = -11.42776870727539
Processing d16  coef = -17.478708267211914
Processing d53  coef = 8.603472709655762
Processing d59  coef = 13.416349411010742
Processing d92  coef = -10.582313537597656
Processing M_relative_abund_sum  coef = 9.958646774291992
Processing d96  coef = 10.235400199890137
Processing d112  coef = -3.063424587249756
Processing d33  coef = 14.322538375854492
Processing d83  coef = -3.1506240367889404
Processing d101  coef = 2.9699225425720215
Processing d24  coef = 2.227816581726074
Processing d78  coef = -19.69312858581543
Processing d39  coef = -9.576157569885254
Processing d35  coef = -20.94719696044922
Processing d18  coef = 13.9391622543334

# RD

In [3]:

# output trait image
out_tif = r"D:\wenqu\2024_data\root_trait_map\site6_RD4_2021.tif"

# =========================================
# 2) SELECTED VARIABLES
# =========================================
var_list = ['d17', 'd72', 'd46', 'd35', 'd59', 'd117', 'd32', 'd96', 'd14',
       'd102', 'd20', 'd6', 'd13', 'd95', 'd81', 'd75', 'd47', 'd110',
       'd44', 'd24', 'F_relative_abund_sum', 'd56', 'd74', 'd86', 'd8']

coefficients = np.array([
    0.0034344 , -0.00738902,  0.0136807 ,  0.00886236, -0.0335986 ,
        0.0043546 ,  0.01245776,  0.00322395, -0.01563085,  0.01063521,
        0.01332692, -0.0077559 , -0.01500148,  0.00154089,  0.00912132,
       -0.00018248,  0.01457909,  0.0128884 , -0.0211283 ,  0.02558462,
        0.01281078, -0.01831331,  0.00956732, -0.01441956,  0.00929774
], dtype=np.float32)

intercept = 0.16445648

if len(var_list) != len(coefficients):
    raise ValueError(f"var_list has {len(var_list)} variables but coefficients has {len(coefficients)} values.")

# =========================================
# 3) PFT FILE NAME MAP
#    Edit the filenames here if needed
# =========================================
pft_map = {
    # "M_relative_abund_sum": os.path.join(pft_folder, "site1b_M.tif"),
    # "G_relative_abund_sum": os.path.join(pft_folder, "site1b_G.tif"),
    "F_relative_abund_sum": os.path.join(pft_folder, "site6_F4_2021.tif"),
}

for k, v in pft_map.items():
    if not os.path.exists(v):
        raise FileNotFoundError(f"Missing PFT tif for {k}: {v}")

# =========================================
# 4) OPEN SITE IMAGE AS REFERENCE
# =========================================
with rasterio.open(site_img) as src:
    print("Site image:", site_img)
    print("Driver:", src.driver)
    print("Band count:", src.count)
    print("Shape:", src.height, src.width)

    ref_meta = src.meta.copy()
    ref_crs = src.crs
    ref_transform = src.transform
    ref_height = src.height
    ref_width = src.width
    ref_nodata = src.nodata

    # use band 1 to create initial valid mask
    ref_band = src.read(1).astype(np.float32)

    if ref_nodata is not None:
        valid_mask = (ref_band != ref_nodata) & np.isfinite(ref_band)
    else:
        valid_mask = np.isfinite(ref_band)

    # initialize output
    result = np.full((ref_height, ref_width), intercept, dtype=np.float32)

    # =========================================
    # 5) LOOP THROUGH VARIABLES
    # =========================================
    for var_name, coef in zip(var_list, coefficients):
        print(f"Processing {var_name}  coef = {coef}")

        # -----------------------------
        # derivative band from site image
        # d41 = b42 - b41
        # -----------------------------
        if var_name.startswith("d"):
            d_idx = int(var_name[1:])   # e.g. 41
            b1 = d_idx
            b2 = d_idx + 1

            if b1 < 1 or b2 > src.count:
                raise ValueError(f"{var_name} needs bands {b1} and {b2}, but site image has only {src.count} bands.")

            arr1 = src.read(b1).astype(np.float32)
            arr2 = src.read(b2).astype(np.float32)

            deriv = arr2 - arr1

            if ref_nodata is not None:
                band_mask = (arr1 != ref_nodata) & (arr2 != ref_nodata)
            else:
                band_mask = np.isfinite(arr1) & np.isfinite(arr2)

            band_mask &= np.isfinite(deriv)
            valid_mask &= band_mask

            result += deriv * coef

        # -----------------------------
        # PFT raster
        # -----------------------------
        else:
            tif_path = pft_map[var_name]

            with rasterio.open(tif_path) as pft_src:
                pft_arr = pft_src.read(1).astype(np.float32)
                pft_nodata = pft_src.nodata

                if pft_nodata is not None:
                    pft_arr[pft_arr == pft_nodata] = np.nan

                dst_arr = np.full((ref_height, ref_width), np.nan, dtype=np.float32)

                reproject(
                    source=pft_arr,
                    destination=dst_arr,
                    src_transform=pft_src.transform,
                    src_crs=pft_src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                band_mask = np.isfinite(dst_arr)
                valid_mask &= band_mask

                result += np.nan_to_num(dst_arr, nan=0.0) * coef

# =========================================
# 6) SET OUTPUT NODATA
# =========================================
out_nodata = -9999.0
result[~valid_mask] = out_nodata

# =========================================
# 7) WRITE OUTPUT
# =========================================
out_meta = ref_meta.copy()
out_meta.update({
    "driver": "GTiff",
    "count": 1,
    "dtype": "float32",
    "nodata": out_nodata
})

os.makedirs(os.path.dirname(out_tif), exist_ok=True)

with rasterio.open(out_tif, "w", **out_meta) as dst:
    dst.write(result, 1)

print("Done.")
print("Output saved to:", out_tif)

Site image: E:\wenqu\2024_data\site6\site6_simulation_4
Driver: ENVI
Band count: 122
Shape: 3152 3052
Processing d17  coef = 0.003434400074183941
Processing d72  coef = -0.007389020174741745
Processing d46  coef = 0.013680700212717056
Processing d35  coef = 0.008862360380589962
Processing d59  coef = -0.03359860181808472
Processing d117  coef = 0.004354599863290787
Processing d32  coef = 0.01245776005089283
Processing d96  coef = 0.003223950043320656
Processing d14  coef = -0.015630850568413734
Processing d102  coef = 0.010635210201144218
Processing d20  coef = 0.013326919637620449
Processing d6  coef = -0.007755899801850319
Processing d13  coef = -0.015001479536294937
Processing d95  coef = 0.0015408899635076523
Processing d81  coef = 0.009121320210397243
Processing d75  coef = -0.00018247999832965434
Processing d47  coef = 0.014579090289771557
Processing d110  coef = 0.012888399884104729
Processing d44  coef = -0.021128300577402115
Processing d24  coef = 0.025584619492292404
Processi

# SRA

In [4]:

# output trait image
out_tif = r"D:\wenqu\2024_data\root_trait_map\site6_SRA4_2021.tif"

# =========================================
# 2) SELECTED VARIABLES
# =========================================
var_list = ['d113', 'd73', 'd87', 'd10', 'd68', 'd14', 'd8', 'd106', 'd53',
       'd69', 'd117', 'd30', 'd118', 'F_relative_abund_sum', 'd35', 'd89',
       'd33', 'd18', 'd22', 'd25', 'd37', 'd100', 'd83', 'd20', 'd58',
       'd19', 'G_relative_abund_sum', 'd112', 'd110', 'd34', 'd115',
       'd86', 'd96', 'd91', 'd74', 'd16', 'd97', 'd32', 'd57', 'd75',
       'd78', 'd108', 'd46', 'd6', 'd39', 'd52']

coefficients = np.array([
 -28.33677044,  14.9841428 ,  39.05242864,  10.93103811,
        -7.55854321,   8.59314491,  26.80935598,  19.05357301,
       -14.99280106,  37.84418612, -11.48677606, -12.31517166,
        41.74143677,  -8.45586796,   6.3018359 ,  12.06967968,
        11.77327899,  10.6486303 , -18.34853025, -55.1526429 ,
        31.47901632,  49.4589139 , -43.43842145,  45.52903614,
        30.07457028,  16.7742977 ,  38.70811979, -23.78154973,
       -47.67027481,  11.58098336, -20.82336703,  42.47031469,
        44.11516315, -59.11566652, -52.68189978, -34.58236979,
         1.82920284, -88.6111718 ,  34.61281766,  72.61304465,
       -72.18462701,  70.56173522, -61.0313768 ,  66.92531262,
       -57.99379916,  58.60693083
], dtype=np.float32)

intercept = 555.51318751

if len(var_list) != len(coefficients):
    raise ValueError(f"var_list has {len(var_list)} variables but coefficients has {len(coefficients)} values.")

# =========================================
# 3) PFT FILE NAME MAP
#    Edit the filenames here if needed
# =========================================
pft_map = {
    # "M_relative_abund_sum": os.path.join(pft_folder, "site1b_M.tif"),
    "G_relative_abund_sum": os.path.join(pft_folder, "site6_G4_2021.tif"),
    "F_relative_abund_sum": os.path.join(pft_folder, "site6_F4_2021.tif"),
}

for k, v in pft_map.items():
    if not os.path.exists(v):
        raise FileNotFoundError(f"Missing PFT tif for {k}: {v}")

# =========================================
# 4) OPEN SITE IMAGE AS REFERENCE
# =========================================
with rasterio.open(site_img) as src:
    print("Site image:", site_img)
    print("Driver:", src.driver)
    print("Band count:", src.count)
    print("Shape:", src.height, src.width)

    ref_meta = src.meta.copy()
    ref_crs = src.crs
    ref_transform = src.transform
    ref_height = src.height
    ref_width = src.width
    ref_nodata = src.nodata

    # use band 1 to create initial valid mask
    ref_band = src.read(1).astype(np.float32)

    if ref_nodata is not None:
        valid_mask = (ref_band != ref_nodata) & np.isfinite(ref_band)
    else:
        valid_mask = np.isfinite(ref_band)

    # initialize output
    result = np.full((ref_height, ref_width), intercept, dtype=np.float32)

    # =========================================
    # 5) LOOP THROUGH VARIABLES
    # =========================================
    for var_name, coef in zip(var_list, coefficients):
        print(f"Processing {var_name}  coef = {coef}")

        # -----------------------------
        # derivative band from site image
        # d41 = b42 - b41
        # -----------------------------
        if var_name.startswith("d"):
            d_idx = int(var_name[1:])   # e.g. 41
            b1 = d_idx
            b2 = d_idx + 1

            if b1 < 1 or b2 > src.count:
                raise ValueError(f"{var_name} needs bands {b1} and {b2}, but site image has only {src.count} bands.")

            arr1 = src.read(b1).astype(np.float32)
            arr2 = src.read(b2).astype(np.float32)

            deriv = arr2 - arr1

            if ref_nodata is not None:
                band_mask = (arr1 != ref_nodata) & (arr2 != ref_nodata)
            else:
                band_mask = np.isfinite(arr1) & np.isfinite(arr2)

            band_mask &= np.isfinite(deriv)
            valid_mask &= band_mask

            result += deriv * coef

        # -----------------------------
        # PFT raster
        # -----------------------------
        else:
            tif_path = pft_map[var_name]

            with rasterio.open(tif_path) as pft_src:
                pft_arr = pft_src.read(1).astype(np.float32)
                pft_nodata = pft_src.nodata

                if pft_nodata is not None:
                    pft_arr[pft_arr == pft_nodata] = np.nan

                dst_arr = np.full((ref_height, ref_width), np.nan, dtype=np.float32)

                reproject(
                    source=pft_arr,
                    destination=dst_arr,
                    src_transform=pft_src.transform,
                    src_crs=pft_src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                band_mask = np.isfinite(dst_arr)
                valid_mask &= band_mask

                result += np.nan_to_num(dst_arr, nan=0.0) * coef

# =========================================
# 6) SET OUTPUT NODATA
# =========================================
out_nodata = -9999.0
result[~valid_mask] = out_nodata

# =========================================
# 7) WRITE OUTPUT
# =========================================
out_meta = ref_meta.copy()
out_meta.update({
    "driver": "GTiff",
    "count": 1,
    "dtype": "float32",
    "nodata": out_nodata
})

os.makedirs(os.path.dirname(out_tif), exist_ok=True)

with rasterio.open(out_tif, "w", **out_meta) as dst:
    dst.write(result, 1)

print("Done.")
print("Output saved to:", out_tif)

Site image: E:\wenqu\2024_data\site6\site6_simulation_4
Driver: ENVI
Band count: 122
Shape: 3152 3052
Processing d113  coef = -28.33677101135254
Processing d73  coef = 14.984143257141113
Processing d87  coef = 39.05242919921875
Processing d10  coef = 10.931037902832031
Processing d68  coef = -7.5585432052612305
Processing d14  coef = 8.593145370483398
Processing d8  coef = 26.809356689453125
Processing d106  coef = 19.053573608398438
Processing d53  coef = -14.99280071258545
Processing d69  coef = 37.84418487548828
Processing d117  coef = -11.486776351928711
Processing d30  coef = -12.315171241760254
Processing d118  coef = 41.74143600463867
Processing F_relative_abund_sum  coef = -8.455867767333984
Processing d35  coef = 6.301836013793945
Processing d89  coef = 12.069679260253906
Processing d33  coef = 11.773279190063477
Processing d18  coef = 10.648630142211914
Processing d22  coef = -18.348529815673828
Processing d25  coef = -55.15264129638672
Processing d37  coef = 31.4790172576904

# RBD

In [5]:
# 'b1', 'b2',, 1.19415731e-03,  5.41468712e-04,

In [5]:

# output trait image
out_tif = r"D:\wenqu\2024_data\root_trait_map\site6_RBD_test4_2021.tif"

# =========================================
# 2) SELECTED VARIABLES
# =========================================
var_list = ['d113', 'd33', 'd61', 'd9',  'd10', 'd45', 'd49', 'd68',
       'd25', 'd93', 'd100', 'd91', 'd104', 'd120', 'd79', 'd90', 'd56',
       'd72', 'd32', 'd5', 'd47', 'd69', 'd96', 'd92', 'd82', 'd7', 'd11',
       'M_relative_abund_sum', 'd106', 'd114', 'd87', 'd29', 'd81', 'd71',
       'd88', 'd18', 'd94', 'd30', 'd84', 'd50', 'd28', 'd2', 'd34',
       'd118', 'd44', 'd12', 'd102', 'd78', 'F_relative_abund_sum', 'd6']

coefficients = np.array([
        1.34848019e-03, -1.38109964e-03,  2.29227662e-03,  1.91560066e-04,
          3.99914562e-04, -2.73061055e-04,
        1.08762535e-03, -9.09175446e-05,  1.20181227e-03, -7.59276447e-04,
        7.73449394e-04,  6.94724051e-05,  6.80494543e-05,  3.15098593e-04,
        1.44200328e-03,  1.79321089e-04,  3.36773205e-04,  2.47976544e-04,
       -5.48878666e-05, -5.19847962e-04,  8.94435817e-04, -3.64472203e-04,
        1.01098884e-03, -1.43307671e-03,  7.73818922e-04, -1.07381929e-03,
        1.14981281e-03,  6.47109347e-04,  1.27535014e-03,  7.24049001e-04,
        2.05208410e-04, -6.61888940e-04, -2.17345024e-03,  1.22285949e-03,
       -2.09095766e-03, -1.02014445e-03, -1.86573155e-03, -1.18400502e-03,
        1.26666770e-03, -1.95644679e-03, -1.52293323e-03, -1.01928707e-03,
       -1.24420432e-03, -9.21418366e-04,  1.65952156e-03,  7.76047402e-04,
       -4.65043151e-04,  1.49684223e-03, -1.65666812e-03, -1.30718668e-03
], dtype=np.float32)

intercept = 0.00550475

if len(var_list) != len(coefficients):
    raise ValueError(f"var_list has {len(var_list)} variables but coefficients has {len(coefficients)} values.")

# =========================================
# 3) PFT FILE NAME MAP
#    Edit the filenames here if needed
# =========================================
pft_map = {
    "M_relative_abund_sum": os.path.join(pft_folder, "site6_M4_2021.tif"),
    # "G_relative_abund_sum": os.path.join(pft_folder, "site2a_G.tif"),
    "F_relative_abund_sum": os.path.join(pft_folder, "site6_F4_2021.tif"),
}

for k, v in pft_map.items():
    if not os.path.exists(v):
        raise FileNotFoundError(f"Missing PFT tif for {k}: {v}")

# =========================================
# 4) OPEN SITE IMAGE AS REFERENCE
# =========================================
with rasterio.open(site_img) as src:
    print("Site image:", site_img)
    print("Driver:", src.driver)
    print("Band count:", src.count)
    print("Shape:", src.height, src.width)

    ref_meta = src.meta.copy()
    ref_crs = src.crs
    ref_transform = src.transform
    ref_height = src.height
    ref_width = src.width
    ref_nodata = src.nodata

    # use band 1 to create initial valid mask
    ref_band = src.read(1).astype(np.float32)

    if ref_nodata is not None:
        valid_mask = (ref_band != ref_nodata) & np.isfinite(ref_band)
    else:
        valid_mask = np.isfinite(ref_band)

    # initialize output
    result = np.full((ref_height, ref_width), intercept, dtype=np.float32)

    # =========================================
    # 5) LOOP THROUGH VARIABLES
    # =========================================
    for var_name, coef in zip(var_list, coefficients):
        print(f"Processing {var_name}  coef = {coef}")

        # -----------------------------
        # derivative band from site image
        # d41 = b42 - b41
        # -----------------------------
        if var_name.startswith("d"):
            d_idx = int(var_name[1:])   # e.g. 41
            b1 = d_idx
            b2 = d_idx + 1

            if b1 < 1 or b2 > src.count:
                raise ValueError(f"{var_name} needs bands {b1} and {b2}, but site image has only {src.count} bands.")

            arr1 = src.read(b1).astype(np.float32)
            arr2 = src.read(b2).astype(np.float32)

            deriv = arr2 - arr1

            if ref_nodata is not None:
                band_mask = (arr1 != ref_nodata) & (arr2 != ref_nodata)
            else:
                band_mask = np.isfinite(arr1) & np.isfinite(arr2)

            band_mask &= np.isfinite(deriv)
            valid_mask &= band_mask

            result += deriv * coef

        # -----------------------------
        # PFT raster
        # -----------------------------
        else:
            tif_path = pft_map[var_name]

            with rasterio.open(tif_path) as pft_src:
                pft_arr = pft_src.read(1).astype(np.float32)
                pft_nodata = pft_src.nodata

                if pft_nodata is not None:
                    pft_arr[pft_arr == pft_nodata] = np.nan

                dst_arr = np.full((ref_height, ref_width), np.nan, dtype=np.float32)

                reproject(
                    source=pft_arr,
                    destination=dst_arr,
                    src_transform=pft_src.transform,
                    src_crs=pft_src.crs,
                    dst_transform=ref_transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                band_mask = np.isfinite(dst_arr)
                valid_mask &= band_mask

                result += np.nan_to_num(dst_arr, nan=0.0) * coef

# =========================================
# 6) SET OUTPUT NODATA
# =========================================
out_nodata = -9999.0
result[~valid_mask] = out_nodata

# =========================================
# 7) WRITE OUTPUT
# =========================================
out_meta = ref_meta.copy()
out_meta.update({
    "driver": "GTiff",
    "count": 1,
    "dtype": "float32",
    "nodata": out_nodata
})

os.makedirs(os.path.dirname(out_tif), exist_ok=True)

with rasterio.open(out_tif, "w", **out_meta) as dst:
    dst.write(result, 1)

print("Done.")
print("Output saved to:", out_tif)

Site image: E:\wenqu\2024_data\site6\site6_simulation_4
Driver: ENVI
Band count: 122
Shape: 3152 3052
Processing d113  coef = 0.0013484802329912782
Processing d33  coef = -0.0013810996897518635
Processing d61  coef = 0.002292276592925191
Processing d9  coef = 0.000191560058738105
Processing d10  coef = 0.00039991457015275955
Processing d45  coef = -0.00027306104311719537
Processing d49  coef = 0.0010876253945752978
Processing d68  coef = -9.091754327528179e-05
Processing d25  coef = 0.0012018122943118215
Processing d93  coef = -0.0007592764450237155
Processing d100  coef = 0.0007734493701718748
Processing d91  coef = 6.94724076311104e-05
Processing d104  coef = 6.804945587646216e-05
Processing d120  coef = 0.00031509858672507107
Processing d79  coef = 0.001442003296688199
Processing d90  coef = 0.0001793210831237957
Processing d56  coef = 0.0003367731987964362
Processing d72  coef = 0.00024797653895802796
Processing d32  coef = -5.488786700880155e-05
Processing d5  coef = -0.0005198479

In [6]:
import rasterio
import numpy as np

# ============================
# input files
# ============================
tif1 = r"D:\wenqu\2024_data\root_trait_map\site6_RBD_test4_2021.tif"


out_tif = r"D:\wenqu\2024_data\root_trait_map\site6_RBD4_2021.tif"

# ============================
# read tif1 (single band)
# ============================
with rasterio.open(tif1) as src1:
    img1 = src1.read(1).astype(np.float32)
    profile = src1.profile

# ============================
# read bands from tif2
# ============================
with rasterio.open(site_img) as src2:
    b1 = src2.read(1).astype(np.float32)   # band 1
    b2 = src2.read(2).astype(np.float32)   # band 2

# ============================
# calculation
# ============================
result = img1 + 1.19415731e-03 * b1 + 5.41468712e-04 * b2

# ============================
# write output
# ============================
profile.update(
    dtype=rasterio.float32,
    count=1
)

with rasterio.open(out_tif, "w", **profile) as dst:
    dst.write(result.astype(np.float32), 1)

print("Finished:", out_tif)

Finished: D:\wenqu\2024_data\root_trait_map\site6_RBD4_2021.tif
